# Pipeline 00 – Deterministischer Textbaustein-Baseline

**Zweck:** Reviewer-Baseline. Beantwortet die Frage:  
*Was leistet das LLM über einen regelbasierten Textbaustein-Generator hinaus?*

Gleiche Inputs wie die LLM-Pipelines (lokale SHAP-/EBM-JSONs), gleiche Dreiteilung  
`[VORHERSAGE]` / `[TREIBER]` / `[EMPFEHLUNG]` – vollständig deterministisch, kein API-Call, Kosten $0.

**Erwartetes Bild im LLM-Judge (NB 07):**
- Faithfulness: Template nennt exakte Top-3-Treiber → hoher Score erwartet
- Clarity: Keine kontextuelle Umformulierung, kein narrativer Fluss → niedriger als LLM
- Completeness: Struktur korrekt, Empfehlung aber rein schwellenwertbasiert ohne Kontextualisierung

In [ ]:
from __future__ import annotations

import json
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from utils import INSTANCE_IDS, RESULTS_DIR, EXPLANATIONS_DIR

LOSS_KEY   = 'poisson_log'
XAI_MODELS = ['xgb', 'ebm']

print(f'Instanzen: {INSTANCE_IDS}')
print(f'Modelle:   {XAI_MODELS}')
print(f'Output:    {RESULTS_DIR / "pipeline00"}')

In [ ]:
# Identische Mappings wie build_judge_prompt in NB 07
WEEKDAYS = {
    0: 'Sonntag', 1: 'Montag', 2: 'Dienstag', 3: 'Mittwoch',
    4: 'Donnerstag', 5: 'Freitag', 6: 'Samstag',
}
MONTHS = {
    1: 'Januar', 2: 'Februar', 3: 'M\u00e4rz', 4: 'April',
    5: 'Mai', 6: 'Juni', 7: 'Juli', 8: 'August',
    9: 'September', 10: 'Oktober', 11: 'November', 12: 'Dezember',
}
WEATHER = {
    1: 'klar/wenige Wolken',
    2: 'Nebel/bew\u00f6lkt',
    3: 'leichter Regen/Schnee',
    4: 'Starkregen/Gewitter',
}
FEATURE_NAMES = {
    'hr':         'Uhrzeit',
    'temp':       'Temperatur',
    'hum':        'Luftfeuchtigkeit',
    'windspeed':  'Windgeschwindigkeit',
    'yr':         'Jahr',
    'mnth':       'Monat',
    'weekday':    'Wochentag',
    'weathersit': 'Wetterlage',
    'holiday':    'Feiertagsstatus',
}


def readable_value(feature: str, raw: float) -> str:
    """Denormalisiert einen Feature-Rohwert in einen lesbaren String."""
    if feature == 'hr':
        return f'{int(raw):02d}:00 Uhr'
    elif feature == 'temp':
        return f'~{raw * 41:.1f} \u00b0C'
    elif feature == 'hum':
        return f'{raw * 100:.0f} %'
    elif feature == 'windspeed':
        return f'{raw * 67:.1f} km/h'
    elif feature == 'yr':
        return '2011' if int(raw) == 0 else '2012'
    elif feature == 'mnth':
        return MONTHS.get(int(raw), str(int(raw)))
    elif feature == 'weekday':
        return WEEKDAYS.get(int(raw), str(int(raw)))
    elif feature == 'weathersit':
        return WEATHER.get(int(raw), str(int(raw)))
    elif feature == 'holiday':
        return 'Feiertag' if int(raw) == 1 else 'kein Feiertag'
    else:
        return str(raw)


print('Hilfstabellen geladen.')

In [ ]:
def generate_template(data: dict) -> str:
    """
    Deterministischer Textbaustein aus lokalem Erkl\u00e4rungs-JSON.
    Struktur identisch zu LLM-Pipelines: [VORHERSAGE] / [TREIBER] / [EMPFEHLUNG].
    """
    pred   = data['prediction']
    actual = data['y_true']
    fv     = data['feature_values']
    top3   = data['contributions'][:3]

    hour    = int(fv['hr'])
    weekday = WEEKDAYS.get(int(fv['weekday']), str(fv['weekday']))
    month   = MONTHS.get(int(fv['mnth']), str(fv['mnth']))
    year    = '2011' if int(fv['yr']) == 0 else '2012'

    # [VORHERSAGE]
    vorhersage = (
        f'[VORHERSAGE] Das Modell prognostiziert f\u00fcr {weekday}, {month} {year}, '
        f'{hour:02d}:00 Uhr insgesamt {pred:.0f} Fahrradausleihungen '
        f'(tats\u00e4chlicher Wert: {actual:.0f}).'
    )

    # [TREIBER]
    lines = []
    for c in top3:
        feat      = c['feature']
        contrib   = c['contribution']
        val_str   = readable_value(feat, c['value'])
        direction = 'erh\u00f6hend' if contrib > 0 else 'senkend'
        fname     = FEATURE_NAMES.get(feat, feat)
        lines.append(f'\u2022 {fname} ({val_str}): {direction} (Beitrag: {contrib:+.2f})')
    treiber = '[TREIBER] Die drei wichtigsten Einflussfaktoren sind:\n' + '\n'.join(lines)

    # [EMPFEHLUNG]
    if 7 <= hour <= 9:
        tageszeit = 'Morgenberufsverkehr'
    elif 17 <= hour <= 19:
        tageszeit = 'Abendberufsverkehr'
    elif 10 <= hour <= 16:
        tageszeit = 'Tagesbetrieb'
    else:
        tageszeit = 'Schwachlastzeit'

    if pred >= 350:
        masse  = 'Hohe Nachfrage'
        aktion = 'Stellen Sie ausreichend Fahrr\u00e4der und Personal an stark frequentierten Stationen bereit.'
    elif pred >= 150:
        masse  = 'Moderate Nachfrage'
        aktion = 'Halten Sie die regul\u00e4re Betriebsbereitschaft aufrecht.'
    else:
        masse  = 'Geringe Nachfrage'
        aktion = 'Ressourcen k\u00f6nnen gezielt reduziert eingesetzt werden.'

    empfehlung = f'[EMPFEHLUNG] {masse} im {tageszeit}. {aktion}'

    return '\n\n'.join([vorhersage, treiber, empfehlung])


# Smoke-Test
_test = json.loads((EXPLANATIONS_DIR / f'local_xgb_{LOSS_KEY}_inst224.json').read_text())
print(generate_template(_test))

In [ ]:
out_dir = RESULTS_DIR / 'pipeline00'
out_dir.mkdir(exist_ok=True)

for xai in XAI_MODELS:
    for iid in INSTANCE_IDS:
        src  = EXPLANATIONS_DIR / f'local_{xai}_{LOSS_KEY}_inst{iid}.json'
        data = json.loads(src.read_text())

        t0          = time.perf_counter()
        explanation = generate_template(data)
        elapsed     = round(time.perf_counter() - t0, 6)

        record = {
            'explanation':  explanation,
            'elapsed_s':    elapsed,
            'usage':        {'input_tokens': 0, 'output_tokens': 0, 'cache_read_input_tokens': 0},
            'n_tool_calls': 0,
            'y_true':       data['y_true'],
            'prediction':   data['prediction'],
        }

        out_file = out_dir / f'{xai}_inst{iid}.json'
        out_file.write_text(json.dumps(record, indent=2, ensure_ascii=False))
        print(f'  {xai.upper()} inst={iid:4d}: {len(explanation.split()):3d} W\u00f6rter  '
              f'pred={data["prediction"]:.0f}  true={data["y_true"]:.0f}')

print(f'\nFertig. {len(XAI_MODELS) * len(INSTANCE_IDS)} Dateien gespeichert in {out_dir}')